# 모델링

생각해볼 질문들:
- 베이스라인 모델은 무엇으로 할 것인가? (예: 평균값 예측, 단순 선형회귀)
- 비교해볼 모델 후보는? (선형회귀, 랜덤포레스트, XGBoost 등)
- 교차검증(cross-validation) 전략은 어떻게 할 것인가?
- 어떤 모델이 스케일링이 필요하고, 어떤 모델은 필요 없는가?
- 하이퍼파라미터 튜닝은 어떻게 할 것인가? (GridSearch, RandomSearch 등)


## 질문에 대한 답

- **베이스라인:** 01에서 정한 대로 **시간대별 평균 예측**을 베이스라인으로 한다. 참고용으로 전체 평균 예측도 함께 잰다. 목표는 시간대별 평균 RMSE보다 약 20% 낮은 RMSE다.
- **모델 후보:** Ridge(선형회귀), RandomForest, XGBoost. 03에서 시간대와 대여량의 관계가 직선이 아니었으므로 트리 모델이 유리할 것으로 예상한다.
- **교차검증 전략:** holdout을 뺀 train(1,165행)에 5-fold `KFold`(shuffle, `random_state=42`)를 쓴다. id가 시간 순서가 아니고 날짜도 없어서 시계열 분할은 할 수 없다. 모든 모델에 같은 fold를 써서 공정하게 비교한다. 별도의 validation split은 만들지 않는다. 데이터가 적어서 한 번 나눈 검증 세트보다 교차검증이 안정적이다.
- **누수 방지:** 결측 대체(`HourMedianImputer`), 선형 모델용 원핫인코딩·스케일링·1시 결측 대체를 모두 Pipeline 안에 넣는다. `cross_validate`와 `RandomizedSearchCV`가 fold마다 학습 fold로만 fit한다. holdout은 이 노트북에서 열지 않는다.
- **스케일링:** Ridge는 계수에 규제를 걸기 때문에 스케일링이 필요하다. 트리 모델은 값의 순서로만 분할하므로 필요 없다.
- **하이퍼파라미터 튜닝:** `RandomizedSearchCV`로 RandomForest와 XGBoost를 둘 다 튜닝한다. 기본값 성능 차이가 교차검증 표준편차보다 작아서, 기본값만 보고 하나를 고를 근거가 약하기 때문이다. 튜닝 후 교차검증 RMSE가 가장 낮은 모델을 최종 모델로 고른다.

In [1]:
import sys
import joblib
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

sys.path.append('..')
from src.preprocessing import HourMedianImputer

# holdout은 05의 최종 평가에서만 쓰므로 여기서는 train만 불러온다
train = pd.read_csv('../data/processed/train.csv')
WEATHER_COLS = [col for col in train.columns if col.startswith('hour_bef_')]

In [2]:
X = train.drop(columns=['id', 'count'])
y = train['count']

cv = KFold(n_splits=5, shuffle=True, random_state=42)
SCORING = {'RMSE': 'neg_root_mean_squared_error', 'MAE': 'neg_mean_absolute_error', 'MSE': 'neg_mean_squared_error'}

def evaluate(model, X=X, y=y):
    scores = cross_validate(model, X, y, cv=cv, scoring=SCORING)
    result = {name: -scores[f'test_{name}'].mean() for name in SCORING}
    result['RMSE_std'] = scores['test_RMSE'].std()
    return result

X.shape

(1165, 10)

## 베이스라인 모델


In [3]:
# 시간대별 평균 예측 = hour 원핫인코딩만 넣은 선형회귀 (예측값이 학습 fold의 시간대별 평균과 같음)
hour_mean = make_pipeline(
    ColumnTransformer([('hour', OneHotEncoder(handle_unknown='ignore'), ['hour'])]),
    LinearRegression(),
)
baseline = pd.DataFrame({
    '전체 평균': evaluate(DummyRegressor()),
    '시간대별 평균': evaluate(hour_mean),
}).T.round(1)

TARGET_RMSE = baseline.loc['시간대별 평균', 'RMSE'] * 0.8
print(f'목표 RMSE (시간대별 평균 대비 20% 개선): {TARGET_RMSE:.1f}')
baseline

목표 RMSE (시간대별 평균 대비 20% 개선): 40.8


,RMSE,MAE,MSE,RMSE_std
전체 평균,81.6,64.6,6689.0,5.1
시간대별 평균,51.0,35.0,2613.2,3.4


## 모델 비교


In [4]:
def make_linear(cols):
    # 1시의 오존·미세먼지는 HourMedianImputer 뒤에도 결측이라 선형 모델용으로 한 번 더 중앙값 대체
    num_cols = [col for col in cols if col != 'hour']
    return make_pipeline(
        HourMedianImputer(WEATHER_COLS),
        ColumnTransformer([
            ('hour', OneHotEncoder(handle_unknown='ignore'), ['hour']),
            ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols),
        ]),
        Ridge(),
    )

def make_rf(**params):
    return make_pipeline(HourMedianImputer(WEATHER_COLS), RandomForestRegressor(random_state=42, n_jobs=-1, **params))

def make_xgb(**params):
    return make_pipeline(HourMedianImputer(WEATHER_COLS), XGBRegressor(random_state=42, n_jobs=-1, **params))

models = {
    'Ridge': make_linear(X.columns),
    'RandomForest': make_rf(n_estimators=300),
    'XGBoost': make_xgb(),
}
comparison = pd.DataFrame({name: evaluate(model) for name, model in models.items()}).T
pd.concat([baseline, comparison]).round(1)

,RMSE,MAE,MSE,RMSE_std
전체 평균,81.6,64.6,6689.0,5.1
시간대별 평균,51.0,35.0,2613.2,3.4
Ridge,42.0,30.5,1773.8,3.1
RandomForest,39.9,26.9,1603.1,2.9
XGBoost,40.7,27.0,1670.8,3.6


### 03 EDA 결과 반영 실험

- `is_peak`: 03에서 이 변수의 구간이 실제 피크와 일부만 맞았다. 넣은 경우와 뺀 경우를 비교한다.
- log1p target: 03에서 로그변환이 분포를 확실히 개선하지 않았다. `TransformedTargetRegressor`로 학습만 로그 단위로 하고 예측은 원래 단위로 되돌려 같은 기준으로 비교한다.

In [5]:
X_no_peak = X.drop(columns='is_peak')
experiments = {}
for name, make in [('RandomForest', lambda: make_rf(n_estimators=300)), ('XGBoost', make_xgb)]:
    experiments[f'{name} (기본)'] = evaluate(make())
    experiments[f'{name} (is_peak 제외)'] = evaluate(make(), X_no_peak)
    experiments[f'{name} (log1p target)'] = evaluate(
        TransformedTargetRegressor(make(), func=np.log1p, inverse_func=np.expm1))
pd.DataFrame(experiments).T.round(1)

,RMSE,MAE,MSE,RMSE_std
RandomForest (기본),39.9,26.9,1603.1,2.9
RandomForest (is_peak 제외),39.7,26.7,1582.6,2.5
RandomForest (log1p target),39.9,26.3,1603.0,3.5
XGBoost (기본),40.7,27.0,1670.8,3.6
XGBoost (is_peak 제외),40.4,27.0,1644.7,3.9
XGBoost (log1p target),40.7,27.2,1674.4,3.9


**실험 결과**

- 기본 비교에서 트리 모델(RandomForest 39.9, XGBoost 40.7)이 Ridge(42.0)보다 낫고, 셋 다 시간대별 평균(51.0)보다 크게 낫다. 다만 RandomForest와 XGBoost의 차이(0.8)는 fold 간 표준편차(약 3)보다 작다.
- `is_peak`를 빼면 두 모델 모두 RMSE가 0.2~0.3 낮아진다. 차이는 작지만, `hour`가 같은 정보를 더 정확히 담고 있으므로 **`is_peak`는 쓰지 않는다**(피처가 적은 쪽을 고름).
- log1p target은 RMSE를 개선하지 않는다(RandomForest의 MAE만 0.6 낮아짐). 주 지표인 RMSE 기준으로 **원래 단위 target을 쓴다**.

## 하이퍼파라미터 튜닝


In [6]:
# 위 실험 결과에 따라 is_peak를 빼고 원래 단위 target으로 튜닝
# 이 환경에서는 XGBoost가 트리 1개당 약 16ms로 느려서 n_iter와 트리 수 범위를 실행 시간(약 15분)에 맞춰 제한했다
X_final = X_no_peak
searches = {
    'RandomForest': RandomizedSearchCV(
        make_rf(),
        {
            'randomforestregressor__n_estimators': randint(200, 500),
            'randomforestregressor__max_depth': [None, 8, 12, 16, 24],
            'randomforestregressor__min_samples_leaf': randint(1, 10),
            'randomforestregressor__max_features': [1.0, 0.7, 0.5, 'sqrt'],
        },
        n_iter=20, cv=cv, scoring='neg_root_mean_squared_error', random_state=42,
    ),
    'XGBoost': RandomizedSearchCV(
        make_xgb(),
        {
            'xgbregressor__n_estimators': randint(100, 500),
            'xgbregressor__learning_rate': [0.02, 0.03, 0.05, 0.1, 0.2],
            'xgbregressor__max_depth': randint(2, 9),
            'xgbregressor__min_child_weight': randint(1, 11),
            'xgbregressor__subsample': uniform(0.6, 0.4),
            'xgbregressor__colsample_bytree': uniform(0.5, 0.5),
            'xgbregressor__reg_lambda': [0.1, 1, 5, 10],
        },
        n_iter=20, cv=cv, scoring='neg_root_mean_squared_error', random_state=42,
    ),
}
for search in searches.values():
    search.fit(X_final, y)

# 튜닝된 설정을 같은 fold로 다시 평가해 세 지표를 함께 본다
tuned = pd.DataFrame({
    f'{name} (튜닝)': evaluate(search.best_estimator_, X_final) for name, search in searches.items()
}).T.round(1)
best_name = tuned['RMSE'].idxmin().replace(' (튜닝)', '')
print('최종 모델:', best_name)
print('최적 파라미터:', searches[best_name].best_params_)
print(f'목표 RMSE {TARGET_RMSE:.1f} 달성:', tuned.loc[f'{best_name} (튜닝)', 'RMSE'] <= TARGET_RMSE)
tuned

최종 모델: XGBoost
최적 파라미터: {'xgbregressor__colsample_bytree': np.float64(0.9744427686266666), 'xgbregressor__learning_rate': 0.1, 'xgbregressor__max_depth': 7, 'xgbregressor__min_child_weight': 2, 'xgbregressor__n_estimators': 364, 'xgbregressor__reg_lambda': 1, 'xgbregressor__subsample': np.float64(0.6390688456025535)}
목표 RMSE 40.8 달성: True


,RMSE,MAE,MSE,RMSE_std
RandomForest (튜닝),39.8,27.1,1591.5,2.7
XGBoost (튜닝),39.0,26.2,1537.3,3.5


**튜닝 결과**

- 튜닝 후 XGBoost가 RMSE 39.0(MAE 26.2)으로 RandomForest(39.8)보다 낮아 최종 모델로 골랐다. 단, 이 차이도 fold 간 표준편차(약 3.5)보다 작아서 두 모델의 성능은 사실상 비슷하다.
- 01의 목표(시간대별 평균 RMSE 51.0 대비 20% 개선 = 40.8 이하)를 교차검증 기준으로 달성했다(약 24% 개선).
- **주의:** 이 점수는 튜닝과 모델 선택에 쓴 바로 그 fold로 잰 것이라 실제보다 낙관적이다. 처음 보는 데이터에서의 성능은 05에서 holdout으로 한 번 잰다.
- MAE 약 26은 평균 대여량(약 109)의 약 24% 수준이다. 03에서 본 것처럼 날짜 정보가 없어 생기는 출근일·주말 차이가 오차의 큰 부분일 것으로 예상되며, 05에서 시간대별 오차로 확인한다.

In [7]:
# RandomizedSearchCV(refit=True)가 최적 설정으로 train 전체에 다시 학습한 모델을 저장
final_model = searches[best_name].best_estimator_
joblib.dump({'model': final_model, 'features': list(X_final.columns)}, '../models/final_model.joblib')
print('저장 완료:', best_name, list(X_final.columns))

저장 완료: XGBoost ['hour', 'hour_bef_temperature', 'hour_bef_precipitation', 'hour_bef_windspeed', 'hour_bef_humidity', 'hour_bef_visibility', 'hour_bef_ozone', 'hour_bef_pm10', 'hour_bef_pm2.5']
